# w0 — Curated Phrase List and Underscore Tokenization

**Project**: Making Taste Legible: Symbolic Boundaries and Expert Valuation in Whisky Reviews  
**Purpose**: Build a curated phrase list of multi-word tasting expressions, apply three-step text normalization to all six text columns, and produce the tokenized dataset that all downstream NLP work (dictionary, embeddings, bigrams) depends on.

## Workflow Overview

| Phase | Steps | What happens |
|-------|-------|--------------|
| **Phase 1** | 1–8 | Load data → review prior list → build curated phrases → hyphen→underscore → phrase tokenization → lemmatization → collocation scan → validation |
| **⏸ PAUSE** | — | Review collocation candidates from Step 7; add approved ones to `additional_phrases` in Phase 2 |
| **Phase 2** | 9–11 | Add optional phrases → finalize phrase list → save outputs |

## Outputs

- `data/whiskyfun_phrases.json` — curated phrase list (40–50 entries)
- `data/whiskyfun_tokenized.parquet` — full dataset with phrase-tokenized text columns + `review_text_original`

## Processing Pipeline

```
Original text
  → Step 4: Hyphen-to-underscore  ("old-school" → "old_school")
  → Step 5: Phrase tokenization   ("tropical fruit" → "tropical_fruit")
  → Step 6: Plural lemmatization  ("berries" → "berry", skips underscore tokens)
  → Final tokenized text
```

---
# Phase 1 — Build, Normalize, Scan
---

## Step 1 — Setup and Data Loading

In [1]:
import re
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import nltk
from nltk.stem import WordNetLemmatizer
from gensim.models.phrases import Phrases, ENGLISH_CONNECTOR_WORDS

nltk.download('wordnet',  quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR       = Path('../data')
PIPELINE_DIR   = Path('../pipeline')
CSV_PATH       = DATA_DIR / 'whiskyfun_analytical_dataset.csv'
PHRASES_PATH   = DATA_DIR / 'whiskyfun_phrases.json'
TOKENIZED_PATH = DATA_DIR / 'whiskyfun_tokenized.parquet'
BIGRAM_TXT     = PIPELINE_DIR / 'bi_gram_list.txt'

TEXT_COLS = ['review_text', 'nose', 'mouth', 'finish', 'comments', 'nmf']

# ---------------------------------------------------------------------------
# Load dataset
# ---------------------------------------------------------------------------
assert CSV_PATH.exists(), f"Missing input: {CSV_PATH}"
df = pd.read_csv(CSV_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print("\nNull counts in text columns:")
for col in TEXT_COLS:
    if col in df.columns:
        n_null = df[col].isna().sum()
        print(f"  {col:<20} {n_null:>6,} nulls  ({n_null/len(df)*100:.1f}%)")

Dataset shape: 11,149 rows × 16 columns

Columns: ['dedupe_hash', 'whisky_name_raw', 'distillery', 'score', 'review_date', 'review_year', 'source_url', 'review_text', 'identity_status', 'match_source', 'review_length', 'nose', 'mouth', 'finish', 'comments', 'nmf']

Null counts in text columns:
  review_text               0 nulls  (0.0%)
  nose                     16 nulls  (0.1%)
  mouth                    19 nulls  (0.2%)
  finish                   29 nulls  (0.3%)
  comments                 49 nulls  (0.4%)
  nmf                      16 nulls  (0.1%)


## Step 2 — Review `pipeline/bi_gram_list.txt`

The pipeline contains a prior manually-drafted bigram list (26 entries). We review it to identify domain-relevant additions to incorporate into the curated phrase list.

In [2]:
assert BIGRAM_TXT.exists(), f"Missing: {BIGRAM_TXT}"
with open(BIGRAM_TXT) as f:
    raw_bigrams = [line.strip() for line in f if line.strip()]

print(f"Entries in bi_gram_list.txt ({len(raw_bigrams)}):")
for entry in raw_bigrams:
    print(f"  {entry}")

Entries in bi_gram_list.txt (26):
  tropical fruit
  cask strength
  dried fruit
  dried fig
  dried citrus
  engine oil
  lamp oil
  single cask
  first fill
  1st fill
  second fill
  2nd fill
  stewed fruit
  stone fruit
  dark fruit
  red fruit
  dark berry
  red berry
  lemon zest
  cocoa powder
  old bottle effect
  good old day
  floor malting
  worm tub
  direct fired
  old book


**Review decisions for `bi_gram_list.txt`**:

| Entry | Decision | Reason |
|-------|----------|--------|
| `tropical fruit` | Already in seed | — |
| `dried fruit` | Already in seed | — |
| `stone fruit` | Already in seed | — |
| `first fill` | Already in seed | — |
| `floor malting` | Already in seed | — |
| `old bottle effect` | Already in seed | — |
| `engine oil` | **Include** | Distinct off-note / flaw descriptor |
| `lamp oil` | **Include** | Distinct off-note / flaw descriptor |
| `cocoa powder` | **Include** | Specific confection/texture note |
| `lemon zest` | **Include** | Specific citrus aromatic |
| `old book` | **Include** | Distinctive oxidative/musty descriptor |
| `worm tub` | **Include** | Production method that appears in tasting notes |
| `direct fired` | **Include** | Production method (distillation style) that affects flavour |
| `stewed fruit` | **Include** | Distinct fruit character (cooked vs fresh) |
| `dark fruit` | **Include** | Broad category descriptor used frequently |
| `red fruit` | **Include** | Broad category descriptor |
| `dark berry` | **Include** | Specific fruit category |
| `red berry` | **Include** | Specific fruit category |
| `dried fig` | **Include** | Specific dried fruit note |
| `dried citrus` | **Include** | Specific dried citrus note |
| `cask strength` | **Skip** | Bottle metadata, not a sensory descriptor |
| `single cask` | **Skip** | Bottle metadata |
| `1st fill` | **Skip** | Non-standard form; `first_fill` already in seed |
| `2nd fill` | **Skip** | Non-standard form |
| `second fill` | **Skip** | Metadata; borderline but seed has `refill_cask` |
| `good old day` | **Skip** | Generic nostalgia phrase, not a tasting descriptor |

## Step 3 — Build Curated Phrase List

33 seed phrases (from the project spec) + 14 additions from `bi_gram_list.txt` = 47 initial curated phrases.

In [3]:
# ---------------------------------------------------------------------------
# Seed phrases (33 — must all be included)
# ---------------------------------------------------------------------------
SEED_PHRASES = [
    # Fruit / aromatics
    'tropical_fruit', 'passion_fruit', 'citrus_fruit', 'stone_fruit',
    'orchard_fruit', 'dried_fruit', 'orange_peel', 'lemon_peel',
    # Peat / smoke / coastal
    'peat_smoke', 'sea_air',
    # Sherry / rancio / oxidative
    'pipe_tobacco',
    # Oak / cask / wood
    'pencil_shavings',
    # Flaws / off-notes
    'wet_cardboard', 'burnt_rubber', 'nail_polish', 'furniture_polish',
    # Finish / complexity
    'long_finish', 'short_finish',
    # Chocolate / confection
    'dark_chocolate', 'milk_chocolate', 'brown_sugar', 'barley_sugar', 'cane_sugar',
    # Tea
    'black_tea', 'green_tea', 'earl_grey', 'pu_erh',
    # Style / nostalgia / production
    'old_school', 'old_bottle_effect', 'floor_malting', 'toasted_bread', 'liquid_smoke',
    # Cask terms
    'sherry_cask', 'bourbon_cask', 'first_fill', 'refill_cask',
]

# ---------------------------------------------------------------------------
# Additions from bi_gram_list.txt (14 domain-relevant entries)
# ---------------------------------------------------------------------------
BIGRAM_LIST_ADDITIONS = [
    'engine_oil', 'lamp_oil',
    'cocoa_powder', 'lemon_zest',
    'old_book', 'worm_tub', 'direct_fired',
    'stewed_fruit', 'dark_fruit', 'red_fruit',
    'dark_berry', 'red_berry',
    'dried_fig', 'dried_citrus',
]

# Combine, dedupe, preserve order (seed first)
seen = set()
curated_phrases = []
for p in SEED_PHRASES + BIGRAM_LIST_ADDITIONS:
    if p not in seen:
        curated_phrases.append(p)
        seen.add(p)

print(f"Curated phrase count: {len(curated_phrases)}")
print(f"  Seed:        {len(SEED_PHRASES)}")
print(f"  Additions:   {len(BIGRAM_LIST_ADDITIONS)}")

# ---------------------------------------------------------------------------
# Corpus frequency in raw review_text (before any preprocessing)
# ---------------------------------------------------------------------------
print(f"\n{'phrase':<35} {'reviews (raw)':>14}")
print('-' * 52)
for phrase in curated_phrases:
    space_form = phrase.replace('_', ' ')
    # Also check hyphenated form
    hyphen_form = phrase.replace('_', '-')
    n = df['review_text'].str.contains(
        r'\b(?:' + re.escape(space_form) + r'|' + re.escape(hyphen_form) + r')(?:e?s)?\b',
        case=False, na=False, regex=True
    ).sum()
    print(f"{phrase:<35} {n:>14,}")

Curated phrase count: 50
  Seed:        36
  Additions:   14

phrase                               reviews (raw)
----------------------------------------------------
tropical_fruit                                 393
passion_fruit                                  363
citrus_fruit                                   230
stone_fruit                                     73
orchard_fruit                                  151
dried_fruit                                    316
orange_peel                                    136
lemon_peel                                     214
peat_smoke                                     445
sea_air                                        103
pipe_tobacco                                   468
pencil_shavings                                175
wet_cardboard                                   15
burnt_rubber                                    14
nail_polish                                     57
furniture_polish                                78
long_finish       

## Step 4 — Hyphen-to-Underscore Conversion

Convert hyphens **between alphabetic characters** to underscores. This treats hyphenated compounds (`old-school`, `over-oaked`, `non-peated`) as single tokens rather than splitting them. Also normalise smart quotes.

**Order matters**: this step runs BEFORE phrase tokenisation. Hyphenated forms are caught here; space-separated forms are caught in Step 5. Both converge to the same underscore token.

In [4]:
# Save originals before any transformation
review_text_original = df['review_text'].copy()

def hyphen_to_underscore(text):
    """Convert hyphens between letters to underscores; normalise smart quotes."""
    if pd.isna(text):
        return ''
    # Smart quotes / typographic characters
    text = (text
            .replace('\u2019', "'")
            .replace('\u2018', "'")
            .replace('\u201c', '"')
            .replace('\u201d', '"')
            .replace('\u2013', '-')
            .replace('\u2014', ' '))
    # Spelling normalisations (applied before phrase tokenisation)
    SPELLING_CORRECTIONS = {
        r'\bpu[\s_-]?ehr\b': 'pu erh',
        r'\bpu[\s_-]?her\b': 'pu erh',
    }
    for pattern, replacement in SPELLING_CORRECTIONS.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    # Hyphens between letters only (not dates like 2012-2025)
    return re.sub(r'(?<=[a-zA-Z])-(?=[a-zA-Z])', '_', text)

# Apply to working copy
df_tok = df.copy()
for col in TEXT_COLS:
    if col in df_tok.columns:
        df_tok[col] = df_tok[col].apply(hyphen_to_underscore)

# Spot-check: print 5 sentences that changed
print("=== Hyphen-to-underscore spot-check (5 changed sentences) ===")
mask = df['review_text'].fillna('') != df_tok['review_text'].fillna('')
shown = 0
for idx in df[mask].index:
    print(f"\nRow {idx}:")
    print(f"  BEFORE: {str(df.at[idx, 'review_text'])}")
    print(f"  AFTER:  {str(df_tok.at[idx, 'review_text'])}")
    shown += 1
    if shown >= 5:
        break

print(f"\nTotal rows changed: {mask.sum():,}")


=== Hyphen-to-underscore spot-check (5 changed sentences) ===

Row 0:
  BEFORE: It doesn’t say it’s a Bruichladdich but it cannot be Browmore, can it? Colour: white wine. Nose: the freshest freshness ever and a brininess that’s even more obvious than usual. Much more obvious… So we have sea air, oysters and seaweed, then a flintiness and then the expected fruitiness, with peaches and not-too-ripe melons as well as just a few herbs and some grass. Chives? The whole is really very fresh. Mouth: excessively fruity, starting with tart lemon and lime, then peaches and melons again. A fairly simple but wonderfully chiselled profile that reminds me a bit of the new official Ten, minus the heavier vanilla/oak. All good. Finish: quite long, with more salt again, brine, lemon and a honeyed aftertaste… Comments: very perfect youngish Bruichladdich, nervous and ultra-clean.
  AFTER:  It doesn't say it's a Bruichladdich but it cannot be Browmore, can it? Colour: white wine. Nose: the freshest fresh

## Step 5 — Phrase Tokenization

Replace space-separated occurrences of each curated phrase with its underscore-joined form. Uses word-boundary-aware, case-insensitive regex with optional plural suffix (`(?:e?s)?`) so "tropical fruits" and "tropical fruit" both map to `tropical_fruit`.

Phrases are sorted **longest-first** to prevent partial-match corruption (e.g. `old_bottle_effect` is applied before `old_school`).

In [5]:
def phrase_to_pattern(phrase):
    """Case-insensitive word-boundary pattern; handles both singular and plural last word."""
    words = re.escape(phrase.replace('_', ' ')).replace(r'\ ', r'[\s_-]+')
    return re.compile(
        r'\b' + words + r'(?:e?s)?\b',
        re.IGNORECASE,
    )

def apply_phrases(text, compiled_patterns):
    if pd.isna(text):
        return text
    for phrase, pat in compiled_patterns:
        text = pat.sub(phrase, text)
    return text

# Sort longest-first to prevent partial-match corruption
sorted_phrases = sorted(curated_phrases, key=lambda p: (len(p), p), reverse=True)
patterns = [(p, phrase_to_pattern(p)) for p in sorted_phrases]

for col in TEXT_COLS:
    if col in df_tok.columns:
        df_tok[col] = df_tok[col].apply(lambda t: apply_phrases(t, patterns))

print("Phrase tokenization applied.")

# Spot-check: old_bottle_effect before old_school
obe_count = df_tok['review_text'].str.contains(
    r'\bold_bottle_effect\b', na=False
).sum()
old_school_count = df_tok['review_text'].str.contains(
    r'\bold_school\b', na=False
).sum()
print(f"  old_bottle_effect in review_text: {obe_count:,}")
print(f"  old_school        in review_text: {old_school_count:,}")

# Spot-check: plural form also captured
tf_count = df_tok['review_text'].str.contains(
    r'\btropical_fruit\b', na=False
).sum()
tf_space = df_tok['review_text'].str.contains(
    r'\btropical fruits?\b', case=False, na=False
).sum()
print(f"  tropical_fruit token: {tf_count:,}  |  'tropical fruit(s)' still space-separated: {tf_space}")

Phrase tokenization applied.
  old_bottle_effect in review_text: 31
  old_school        in review_text: 359
  tropical_fruit token: 392  |  'tropical fruit(s)' still space-separated: 0


## Step 6 — Plural Lemmatization (Nouns Only)

Normalise plural nouns to singular using NLTK's WordNetLemmatizer with `pos='n'`. This consolidates vocabulary for dictionary matching and embedding training.

**Skips**: underscore-joined phrase tokens, tokens ≤ 2 characters, non-alphabetic tokens.

**Does NOT change**: adjectives (`smoky`, `peaty`, `waxy`), past participles (`balanced`), or anything already underscore-joined.

In [6]:
lemmatizer = WordNetLemmatizer()
PUNCTUATED_WORD = re.compile(r'^([^A-Za-z]*)([A-Za-z]+)([^A-Za-z]*)$')

def lemmatize_text(text):
    """Lemmatize individual tokens as nouns only; skip underscore-joined phrases."""
    if pd.isna(text):
        return text
    tokens = text.split()
    result = []
    for token in tokens:
        lowered = token.lower()
        if '_' in token:
            result.append(lowered)
            continue
        if token.isalpha():
            result.append(lowered if len(token) <= 2 else lemmatizer.lemmatize(lowered, pos='n'))
            continue
        punctuated_word = PUNCTUATED_WORD.fullmatch(token)
        if punctuated_word and len(punctuated_word.group(2)) > 2:
            prefix, word, suffix = punctuated_word.groups()
            result.append(prefix + lemmatizer.lemmatize(word.lower(), pos='n') + suffix)
        else:
            result.append(lowered)
    return ' '.join(result)

# Collect changed words for validation BEFORE modifying df_tok
changed_examples = []
for text in df_tok['review_text'].dropna().head(500):
    for token in text.split():
        if '_' not in token and len(token) > 2 and token.isalpha():
            lemma = lemmatizer.lemmatize(token.lower(), pos='n')
            if lemma != token.lower():
                changed_examples.append((token, lemma))

# Apply to all text columns
for col in TEXT_COLS:
    if col in df_tok.columns:
        df_tok[col] = df_tok[col].apply(lemmatize_text)

print("Lemmatization applied.")

# Print 20 example changes
seen_pairs = set()
print("\nSample lemmatization changes (before → after):")
for original, lemma in changed_examples:
    pair = (original.lower(), lemma)
    if pair not in seen_pairs:
        print(f"  {original:<20} → {lemma}")
        seen_pairs.add(pair)
    if len(seen_pairs) >= 20:
        break

# Verify adjectives are unchanged
print("\nVerify adjectives unchanged:")
for word in ['smoky', 'peaty', 'waxy', 'balanced', 'lingering']:
    lemma = lemmatizer.lemmatize(word, pos='n')
    status = 'OK (unchanged)' if lemma == word else f'CHANGED → {lemma}'
    print(f"  {word:<20} {status}")

Lemmatization applied.

Sample lemmatization changes (before → after):
  oysters              → oyster
  peaches              → peach
  melons               → melon
  herbs                → herb
  chestnuts            → chestnut
  Goes                 → go
  raisins              → raisin
  whiffs               → whiff
  neglects             → neglect
  monsters             → monster
  whiskies             → whisky
  Charlottes           → charlotte
  bottles              → bottle
  merchants            → merchant
  spices               → spice
  knows                → know
  notes                → note
  prunes               → prune
  touches              → touch
  cloves               → clove

Verify adjectives unchanged:
  smoky                OK (unchanged)
  peaty                OK (unchanged)
  waxy                 OK (unchanged)
  balanced             OK (unchanged)
  lingering            OK (unchanged)


## Step 7 — Collocation Scan for Corpus Expansion

Run `gensim.models.phrases.Phrases` (NPMI scoring) on the tokenized `review_text` to surface additional multi-word candidates beyond the curated list. We apply conservative filters: no already-curated phrases, no `with_water`, no stopword-only pairs, no generic filler.

Present top 50 candidates for user review. Any approved additions go into `additional_phrases` in Phase 2.

In [7]:
def tokenize(text):
    """Lowercase; split on non-alphanumeric, preserving underscores."""
    if pd.isna(text):
        return []
    return re.findall(r'[a-z_]+', str(text).lower())

sentences = [tokenize(t) for t in df_tok['review_text']]
sentences = [s for s in sentences if s]

bigram_model = Phrases(
    sentences,
    min_count=20,
    threshold=0.2,
    connector_words=ENGLISH_CONNECTOR_WORDS,
    scoring='npmi',
)

raw_phrases = bigram_model.export_phrases()
bigram_scores = {}
for k, v in raw_phrases.items():
    key = k.decode() if isinstance(k, bytes) else k
    bigram_scores[key] = float(v)

all_candidates = sorted(bigram_scores.items(), key=lambda x: -x[1])
print(f"Total bigram candidates before filtering: {len(all_candidates):,}")

Total bigram candidates before filtering: 4,246


In [8]:
# ---------------------------------------------------------------------------
# Filters
# ---------------------------------------------------------------------------
nltk_stopwords  = set(stopwords.words('english'))
all_stopwords   = nltk_stopwords | set(ENGLISH_CONNECTOR_WORDS)
curated_set     = set(curated_phrases)

FILLER_PATTERNS = {
    'a_lot', 'of_course', 'very_much', 'would_be', 'i_think',
    'as_well', 'so_far', 'at_least', 'a_bit', 'a_little',
    'quite_nice', 'very_good', 'very_nice', 'pretty_good',
    'not_quite', 'not_bad', 'not_sure', 'really_good',
    'make_sure', 'even_more', 'more_or', 'or_less',
    'in_the', 'on_the', 'of_the', 'at_the', 'to_the',
    'by_the', 'for_the', 'from_the', 'with_the',
    'and_the', 'or_the', 'but_the',
}

def should_filter(bigram):
    if bigram in curated_set:
        return True
    # constituent-word match against curated (handles word-set equality)
    parts = set(bigram.split('_'))
    for phrase in curated_set:
        if parts == set(phrase.split('_')):
            return True
    if 'with_water' in bigram or bigram == 'with_water':
        return True
    ws = bigram.split('_')
    if len(ws) != 2:
        return True
    if ws[0] in all_stopwords and ws[1] in all_stopwords:
        return True
    if bigram in FILLER_PATTERNS:
        return True
    return False

# Build frequency lookup
sent_sets = [set(s) for s in sentences]

filtered_candidates = [
    (bg, score) for bg, score in all_candidates if not should_filter(bg)
]
print(f"After filtering: {len(filtered_candidates):,} candidates remain")

# Print top 50
MAX_DISPLAY = 50
rows = []
for bigram, score in filtered_candidates[:MAX_DISPLAY]:
    freq = sum(1 for s in sent_sets if bigram in s)
    rows.append({'bigram': bigram, 'freq': freq, 'npmi': round(score, 4)})

scan_df = pd.DataFrame(rows).sort_values('freq', ascending=False).reset_index(drop=True)
print(f"\nTop {len(scan_df)} collocation candidates (sorted by corpus frequency):")
print(scan_df.to_string(index=True))

After filtering: 3,584 candidates remain

Top 50 collocation candidates (sorted by corpus frequency):
              bigram  freq    npmi
0     anti_maltoporn   171  1.7471
1         ex_bourbon   129  1.0922
2         triple_sec    87  1.1899
3        ultra_clean    69  0.9844
4      ultra_classic    59  1.0159
5            blade_y    58  0.9835
6        entry_level    36  0.9841
7             jell_o    34  1.1678
8         pot_pourri    34  1.0020
9      grand_marnier    31  1.0330
10           pre_war    31  1.0308
11         ex_peater    30  0.9261
12       extra_years    26  1.1279
13          wham_bam    25  1.0043
14     fernet_branca    22  1.0684
15           chen_pi    19  1.1478
16      tutti_frutti    16  1.0959
17             ho_ho    16  1.0673
18      aston_martin     8  1.0001
19         hong_kong     2  1.0074
20         cod_liver     1  0.9353
21      palo_cortado     1  0.9933
22          caol_ila     1  0.9789
23     travel_retail     1  0.9484
24        au_naturel   

## Step 8 — Validation

Three checks:
1. Full frequency table for all curated phrases in the tokenized corpus.
2. End-to-end spot-check on 5 reviews (original → after hyphen step → after phrase step → after lemma step).
3. Integrity check: space-separated forms of curated phrases should no longer appear.

In [9]:
# 1. Frequency table for curated phrases in tokenized corpus
print(f"{'phrase':<35} {'reviews (tokenized)':>20}")
print('-' * 57)
for phrase in sorted(curated_phrases):
    n = df_tok['review_text'].str.contains(
        r'\b' + re.escape(phrase) + r'\b', na=False
    ).sum()
    print(f"{phrase:<35} {n:>20,}")

phrase                               reviews (tokenized)
---------------------------------------------------------
barley_sugar                                         207
black_tea                                            332
bourbon_cask                                          73
brown_sugar                                           76
burnt_rubber                                          14
cane_sugar                                            26
citrus_fruit                                         229
cocoa_powder                                          97
dark_berry                                             1
dark_chocolate                                       251
dark_fruit                                           199
direct_fired                                           9
dried_citrus                                          18
dried_fig                                            193
dried_fruit                                          315
earl_grey                     

In [10]:
# 2. End-to-end spot-check on 5 reviews
# Re-run the pipeline steps individually on a small sample to show each stage

# Pick 5 reviews that show visible changes across all three steps
mask_changed = df['review_text'].fillna('') != df_tok['review_text'].fillna('')
sample_idx = df[mask_changed].index[:5]

print("=== End-to-end spot-check (5 reviews) ===")
for idx in sample_idx:
    orig  = str(df.at[idx, 'review_text'])
    step2 = hyphen_to_underscore(orig)
    step3 = apply_phrases(step2, patterns)
    step4 = lemmatize_text(step3)
    final = str(df_tok.at[idx, 'review_text'])

    print(f"\nRow {idx}:")
    print(f"  ORIGINAL : {orig[:200]}")
    print(f"  STEP 4   : {step2[:200]}")
    print(f"  STEP 5   : {step3[:200]}")
    print(f"  STEP 6   : {step4[:200]}")
    print(f"  DF_TOK   : {final[:200]}")
    match = '✓ matches' if step4 == final else '✗ MISMATCH'
    print(f"  ({match})")

=== End-to-end spot-check (5 reviews) ===

Row 0:
  ORIGINAL : It doesn’t say it’s a Bruichladdich but it cannot be Browmore, can it? Colour: white wine. Nose: the freshest freshness ever and a brininess that’s even more obvious than usual. Much more obvious… So 
  STEP 4   : It doesn't say it's a Bruichladdich but it cannot be Browmore, can it? Colour: white wine. Nose: the freshest freshness ever and a brininess that's even more obvious than usual. Much more obvious… So 
  STEP 5   : It doesn't say it's a Bruichladdich but it cannot be Browmore, can it? Colour: white wine. Nose: the freshest freshness ever and a brininess that's even more obvious than usual. Much more obvious… So 
  STEP 6   : it doesn't say it's a bruichladdich but it cannot be browmore, can it? colour: white wine. nose: the freshest freshness ever and a brininess that's even more obvious than usual. much more obvious… so 
  DF_TOK   : it doesn't say it's a bruichladdich but it cannot be browmore, can it? colour: wh

### Note: 
Going back to step 5 and 6, re-run them.
Then the validation pass

In [11]:
# 3. Integrity check: space-separated forms should no longer appear
print("=== Integrity check: space-separated forms remaining ===")
print(f"{'phrase':<35} {'space-form remaining':>22}  status")
print('-' * 65)
issues = []
for phrase in sorted(curated_phrases):
    space_form = phrase.replace('_', ' ')
    n_remaining = df_tok['review_text'].str.contains(
        r'\b' + re.escape(space_form) + r'\b', case=False, na=False, regex=True
    ).sum()
    status = 'OK' if n_remaining == 0 else f'WARNING ({n_remaining} remaining)'
    if n_remaining > 0:
        issues.append(phrase)
    print(f"{phrase:<35} {n_remaining:>22,}  {status}")

if issues:
    print(f"\nPhrases with remaining space-separated forms: {issues}")
else:
    print("\nAll curated phrases fully tokenized — no space-separated forms remain.")

=== Integrity check: space-separated forms remaining ===
phrase                                space-form remaining  status
-----------------------------------------------------------------
barley_sugar                                             0  OK
black_tea                                                0  OK
bourbon_cask                                             0  OK
brown_sugar                                              0  OK
burnt_rubber                                             0  OK
cane_sugar                                               0  OK
citrus_fruit                                             0  OK
cocoa_powder                                             0  OK
dark_berry                                               0  OK
dark_chocolate                                           0  OK
dark_fruit                                               0  OK
direct_fired                                             0  OK
dried_citrus                                          

---
# ⏸ PAUSE — Human Review

**Review the collocation candidates in the Step 7 output above.** Decide which (if any) to add to the phrase list.

Guidelines:
- **Add** if the bigram is a genuine semantic unit in whisky tasting vocabulary that you want treated as a single token in the embedding model and dictionary.
- **Skip** if it is generic English (`very_clean`, `quite_light`), structural (`with_water`), or metadata.
- The curated list already has 47 entries; the target ceiling is ~50. Be selective.
- **Note on trigrams**: if you spot interesting three-word combinations in the candidates, note them here but do NOT add them — they are out of scope for this notebook. Pass them to `w0_preproc_bigrams.ipynb` instead.

**After reviewing**, edit `additional_phrases` in Step 9 below, then run Phase 2 from top to bottom.

---

---
# Phase 2 — Add Optional Phrases and Save

> **Edit `additional_phrases` in Step 9, then run all cells in this section.**
---

## Step 9 — Additional Phrases from Collocation Scan

Edit the list below with any phrases you want to add from the Step 7 candidates. Leave empty to keep only the 47 curated phrases.

In [12]:
# ============================================================
#  EDIT THIS LIST — add approved candidates from Step 7
# ============================================================
additional_phrases = [
    # paste approved phrases here, e.g.:
    # "apple_juice",
    # "rubber_note",
    "anti_maltoporn",
    "ex_bourbon",
    "triple_sec",
    "ultra_clean",
    "ultra_classic",
    "blade_y",
    "entry_level",
    "pot_pourri",
    "jell_o",
    "pre_war",
    "grand_marnier",
    "ex_peater",
    "wham_bam",
    "fernet_branca",
    "chen_pi",
    "tutti_frutti",
]
# ============================================================

print(f"Additional phrases: {len(additional_phrases)}")
for p in additional_phrases:
    print(f"  {p}")

Additional phrases: 16
  anti_maltoporn
  ex_bourbon
  triple_sec
  ultra_clean
  ultra_classic
  blade_y
  entry_level
  pot_pourri
  jell_o
  pre_war
  grand_marnier
  ex_peater
  wham_bam
  fernet_branca
  chen_pi
  tutti_frutti


## Step 10 — Apply Additional Phrases and Finalize

In [13]:
if additional_phrases:
    # Validate: no duplicates with curated list
    overlap = [p for p in additional_phrases if p in curated_set]
    if overlap:
        print(f"WARNING: these are already in the curated list and will be skipped: {overlap}")

    new_phrases = [p for p in additional_phrases if p not in curated_set]

    # Sort new phrases longest-first, apply to all columns
    sorted_new = sorted(new_phrases, key=lambda p: (len(p), p), reverse=True)
    new_patterns = [(p, phrase_to_pattern(p)) for p in sorted_new]

    for col in TEXT_COLS:
        if col in df_tok.columns:
            df_tok[col] = df_tok[col].apply(lambda t: apply_phrases(t, new_patterns))

    print(f"Applied {len(new_phrases)} additional phrases to all text columns.")
else:
    new_phrases = []
    print("No additional phrases — using curated list as-is.")

# Build final phrase list (deduplicated, longest-first)
seen = set()
final_phrases = []
for p in sorted(curated_phrases + new_phrases, key=lambda p: (len(p), p), reverse=True):
    if p not in seen:
        final_phrases.append(p)
        seen.add(p)

print(f"\nFinal phrase count: {len(final_phrases)}")

# Updated frequency table
print(f"\n{'phrase':<35} {'reviews':>10}")
print('-' * 47)
for phrase in final_phrases:
    n = df_tok['review_text'].str.contains(
        r'\b' + re.escape(phrase) + r'\b', na=False
    ).sum()
    print(f"{phrase:<35} {n:>10,}")

Applied 16 additional phrases to all text columns.

Final phrase count: 66

phrase                                 reviews
-----------------------------------------------
old_bottle_effect                           31
furniture_polish                            78
pencil_shavings                            175
tropical_fruit                             392
milk_chocolate                             207
dark_chocolate                             251
anti_maltoporn                             172
wet_cardboard                               15
ultra_classic                               61
toasted_bread                               90
passion_fruit                              363
orchard_fruit                              151
grand_marnier                               39
floor_malting                                7
fernet_branca                               65
tutti_frutti                                32
stewed_fruit                                67
short_finish                  

## Step 11 — Save Outputs

In [14]:
# Add review_text_original as the last column
df_tok['review_text_original'] = review_text_original.values

# Confirm shape
print(f"Final dataset shape: {df_tok.shape[0]:,} rows × {df_tok.shape[1]} columns")
assert df_tok.shape[0] == len(df), "Row count mismatch!"
assert 'review_text_original' in df_tok.columns

# Save phrase list
with open(PHRASES_PATH, 'w') as f:
    json.dump(final_phrases, f, indent=2)
print(f"Saved: {PHRASES_PATH}  ({len(final_phrases)} phrases)")

# Save tokenized parquet
df_tok.to_parquet(TOKENIZED_PATH, index=False)
print(f"Saved: {TOKENIZED_PATH}  ({df_tok.shape[0]:,} rows, {df_tok.shape[1]} columns)")

# Verify files exist
assert PHRASES_PATH.exists()
assert TOKENIZED_PATH.exists()
print("\nBoth output files verified.")

Final dataset shape: 11,149 rows × 17 columns
Saved: ../data/whiskyfun_phrases.json  (66 phrases)
Saved: ../data/whiskyfun_tokenized.parquet  (11,149 rows, 17 columns)

Both output files verified.


---
## Acceptance Criteria Checklist

- [ ] `data/whiskyfun_phrases.json` exists with 40–50 entries.
- [ ] `data/whiskyfun_tokenized.parquet` exists with all 11,149 rows, all original columns, plus `review_text_original`.
- [ ] Spot-check: `tropical_fruit` appears as a single token; "tropical fruit" and "tropical fruits" no longer appear as separate words.
- [ ] Spot-check: `old_school` appears for both originally-hyphenated ("old-school") and space-separated ("old school") occurrences.
- [ ] Spot-check: `fruits` → `fruit`, `berries` → `berry` in the lemmatized text, but `smoky`, `peaty`, `waxy` are unchanged.
- [ ] Frequency report printed for all phrases (Step 8 / Step 10).
- [ ] Lemmatization sample (20 changed words) printed and inspected (Step 6).
- [ ] Integrity check: no space-separated forms of curated phrases remain (Step 8).
- [ ] The notebook runs clean from top to bottom.